# CSI 300 Index Return Analysis
**CISC7204 Practice 09 - Data Science Portfolio Project**

This notebook analyses daily returns, volatility, and drawdown of the CSI 300 Index from 2021 to 2025.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Download Data

In [ ]:
TICKER = '000300.SS'
START = '2021-01-01'
END = '2025-12-31'

data = yf.download(TICKER, start=START, end=END)
print(f'Downloaded {len(data)} trading days.')
data.head()

## 2. Compute Daily Log Returns

In [ ]:
data['log_return'] = np.log(data['Close'] / data['Close'].shift(1))
data = data.dropna()

mu = data['log_return'].mean()
sigma = data['log_return'].std()
ann_vol = sigma * np.sqrt(252)

print(f'Mean daily return : {mu:.4%}')
print(f'Daily std dev     : {sigma:.4%}')
print(f'Annualized vol    : {ann_vol:.2%}')

## 3. Return Distribution Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data['log_return'], bins=60, kde=True, color='steelblue', stat='density', ax=ax)

x = np.linspace(data['log_return'].min(), data['log_return'].max(), 200)
pdf = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/sigma)**2)
ax.plot(x, pdf, color='crimson', ls='--', lw=2, label='Normal distribution')

ax.set_title('CSI 300 Daily Log Return Distribution (2021–2025)')
ax.set_xlabel('Daily Log Return')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('figures/daily_returns_histogram.png')
plt.show()

## 4. Rolling 30-Day Volatility

In [ ]:
data['rolling_vol_30d'] = data['log_return'].rolling(30).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(data.index, data['rolling_vol_30d'], color='darkorange', lw=1.5)
ax.axhline(y=ann_vol, color='gray', ls='--', alpha=0.7, label=f'Full-period vol = {ann_vol:.2%}')
ax.set_title('CSI 300 Rolling 30-Day Annualized Volatility')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend()
plt.tight_layout()
plt.savefig('figures/rolling_volatility.png')
plt.show()

## 5. Key Findings

1. Average daily return is slightly negative over 2021–2025, reflecting a prolonged sideways/bear market.
2. Annualized volatility is approximately 19%, with clear spikes during the 2022 downturn and 2024 correction.
3. Return distribution exhibits fat tails — extreme moves occur more often than a normal distribution predicts.

**Limitation**: This analysis does not control for macroeconomic factors (interest rates, policy changes, GDP growth).